<a href="https://colab.research.google.com/github/yolacin/LocalDispersion/blob/main/ToGitHub_Version_2_BestKNNC2SearchForEIAData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.neighbors import NearestNeighbors, KNeighborsRegressor
from sklearn.metrics import mean_absolute_error
from datetime import date, timedelta, datetime

# Mount Google Drive
drive.mount('/content/drive')

# Upload the data:
path_to_file = "/content/drive/My Drive/PEMS_2022_24_TRAFFIC_FLOW_PREDICTION/EIA_DATA/PJM_Cleaned_Hourly_Demand_Dataset_BeforeCovid.csv"
eia_data = pd.read_csv(path_to_file)

# ---------------------------------------------------------
# Step 1: Create 168 Hourly Lags (1 Week of History)
# ---------------------------------------------------------
print("Generating lag features...")
# Ensure data is sorted chronologically
eia_data['UTC time'] = pd.to_datetime(eia_data['timestamp'])
eia_data = eia_data.sort_values('UTC time').reset_index(drop=True)

# Generate lags from 1 hour up to 168 hours (1 week)
lag_features = []
for i in range(1, 169):
    col_name = f'lag_{i}'
    eia_data[col_name] = eia_data['Demand'].shift(i)
    lag_features.append(col_name)

# FIX: Only drop rows where our target or lag features contain NaN values!
columns_to_check = ['Demand'] + lag_features
eia_data = eia_data.dropna(subset=columns_to_check).reset_index(drop=True)

print(f"Dataframe successfully prepared. Total rows available: {len(eia_data)}")

# Separate features (X) and target (y)
X_all_lags = eia_data[lag_features].values
y_all = eia_data['Demand'].values

# ---------------------------------------------------------
# Step 2: Chronological Train-Validation-Test Split
# ---------------------------------------------------------
total_rows = len(eia_data)
train_ratio, val_ratio, test_ratio = 0.65, 0.17, 0.18

# Calculate split index positions
train_end = int(total_rows * train_ratio)
val_end = int(total_rows * (train_ratio + val_ratio))

# Split target values
y_train = y_all[:train_end]
y_val = y_all[train_end:val_end]
y_test = y_all[val_end:]  # Saved for later use, untouched for now

# Split lag feature matrices
X_train_lags = X_all_lags[:train_end]
X_val_lags = X_all_lags[train_end:val_end]

print(f"Data successfully split:")
print(f" - Train rows: {len(y_train)}")
print(f" - Validation rows: {len(y_val)}")
print(f" - Test rows: {len(y_test)}")

# ---------------------------------------------------------
# Step 3: Define Search Parameter Ranges
# ---------------------------------------------------------
w1_range = list(range(2, 15))
K1_range = list(range(2, 15))
w2_range = list(range(2, 15))
K2_range = list(range(2, 15))

# ---------------------------------------------------------
# Step 4: Precompute Validation Predictions
# ---------------------------------------------------------
print("\nPrecomputing short-window model predictions...")
pred_short = {}
for w1 in w1_range:
    # Slice features up to window size w1
    X_tr = X_train_lags[:, :w1]
    X_va = X_val_lags[:, :w1]
    for K1 in K1_range:
        knn = KNeighborsRegressor(n_neighbors=K1, n_jobs=-1)
        knn.fit(X_tr, y_train)
        pred_short[(w1, K1)] = knn.predict(X_va)

print("Precomputing long-window model predictions...")
pred_long = {}
for w2 in w2_range:
    # Slice features up to window size w2
    X_tr = X_train_lags[:, :w2]
    X_va = X_val_lags[:, :w2]
    for K2 in K2_range:
        knn = KNeighborsRegressor(n_neighbors=K2, n_jobs=-1)
        knn.fit(X_tr, y_train)
        pred_long[(w2, K2)] = knn.predict(X_va)

# ---------------------------------------------------------
# Step 5: Evaluate Ensembles to Find the Best Hybrid Pairs
# ---------------------------------------------------------
print(f"\nEvaluating all {len(pred_short) * len(pred_long)} combinations...")
best_mae = float('inf')
best_params = None

# Grid search by combining the precomputed predictions
for (w1, K1), p_short in pred_short.items():
    for (w2, K2), p_long in pred_long.items():

        # Calculate the ensemble prediction (average of both KNNs)
        ensemble_pred = (p_short + p_long) / 2.0

        # Calculate validation Mean Absolute Error using NumPy for raw speed
        mae = np.mean(np.abs(ensemble_pred - y_val))

        if mae < best_mae:
            best_mae = mae
            best_params = (w1, K1, w2, K2)

# ---------------------------------------------------------
# Step 6: Display the Optimization Results
# ---------------------------------------------------------
opt_w1, opt_K1, opt_w2, opt_K2 = best_params

print("\n" + "="*50)
print("GRID SEARCH COMPLETE")
print("="*50)
print(f"Best Validation MAE: {best_mae:.2f}")
print(f"Optimal Short Window (w1): {opt_w1}")
print(f"Optimal Short K-Neighbors (K1): {opt_K1}")
print(f"Optimal Long Window (w2): {opt_w2}")
print(f"Optimal Long K-Neighbors (K2): {opt_K2}")
print("="*50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Generating lag features...


/tmp/ipykernel_699/3050906670.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  eia_data[col_name] = eia_data['Demand'].shift(i)
/tmp/ipykernel_699/3050906670.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  eia_data[col_name] = eia_data['Demand'].shift(i)
/tmp/ipykernel_699/3050906670.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-frag

Dataframe successfully prepared. Total rows available: 29633
Data successfully split:
 - Train rows: 19261
 - Validation rows: 5038
 - Test rows: 5334

Precomputing short-window model predictions...
Precomputing long-window model predictions...

Evaluating all 28561 combinations...

GRID SEARCH COMPLETE
Best Validation MAE: 1032.21
Optimal Short Window (w1): 3
Optimal Short K-Neighbors (K1): 5
Optimal Long Window (w2): 6
Optimal Long K-Neighbors (K2): 4


In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.neighbors import NearestNeighbors, KNeighborsRegressor
from sklearn.metrics import mean_absolute_error
from datetime import date, timedelta, datetime

# Mount Google Drive
drive.mount('/content/drive')

# Upload the data:
path_to_file = "/content/drive/My Drive/PEMS_2022_24_TRAFFIC_FLOW_PREDICTION/EIA_DATA/PJM_Cleaned_Hourly_Demand_Dataset_AfterCovid.csv"
eia_data = pd.read_csv(path_to_file)

# ---------------------------------------------------------
# Step 1: Create 168 Hourly Lags (1 Week of History)
# ---------------------------------------------------------
print("Generating lag features...")
# Ensure data is sorted chronologically
eia_data['UTC time'] = pd.to_datetime(eia_data['timestamp'])
eia_data = eia_data.sort_values('UTC time').reset_index(drop=True)

# Generate lags from 1 hour up to 168 hours (1 week)
lag_features = []
for i in range(1, 169):
    col_name = f'lag_{i}'
    eia_data[col_name] = eia_data['Demand'].shift(i)
    lag_features.append(col_name)

# FIX: Only drop rows where our target or lag features contain NaN values!
columns_to_check = ['Demand'] + lag_features
eia_data = eia_data.dropna(subset=columns_to_check).reset_index(drop=True)

print(f"Dataframe successfully prepared. Total rows available: {len(eia_data)}")

# Separate features (X) and target (y)
X_all_lags = eia_data[lag_features].values
y_all = eia_data['Demand'].values

# ---------------------------------------------------------
# Step 2: Chronological Train-Validation-Test Split
# ---------------------------------------------------------
total_rows = len(eia_data)
train_ratio, val_ratio, test_ratio = 0.65, 0.17, 0.18

# Calculate split index positions
train_end = int(total_rows * train_ratio)
val_end = int(total_rows * (train_ratio + val_ratio))

# Split target values
y_train = y_all[:train_end]
y_val = y_all[train_end:val_end]
y_test = y_all[val_end:]  # Saved for later use, untouched for now

# Split lag feature matrices
X_train_lags = X_all_lags[:train_end]
X_val_lags = X_all_lags[train_end:val_end]

print(f"Data successfully split:")
print(f" - Train rows: {len(y_train)}")
print(f" - Validation rows: {len(y_val)}")
print(f" - Test rows: {len(y_test)}")

# ---------------------------------------------------------
# Step 3: Define Search Parameter Ranges
# ---------------------------------------------------------
w1_range = list(range(2, 15))
K1_range = list(range(2, 15))
w2_range = list(range(2, 15))
K2_range = list(range(2, 15))

# ---------------------------------------------------------
# Step 4: Precompute Validation Predictions
# ---------------------------------------------------------
print("\nPrecomputing short-window model predictions...")
pred_short = {}
for w1 in w1_range:
    # Slice features up to window size w1
    X_tr = X_train_lags[:, :w1]
    X_va = X_val_lags[:, :w1]
    for K1 in K1_range:
        knn = KNeighborsRegressor(n_neighbors=K1, n_jobs=-1)
        knn.fit(X_tr, y_train)
        pred_short[(w1, K1)] = knn.predict(X_va)

print("Precomputing long-window model predictions...")
pred_long = {}
for w2 in w2_range:
    # Slice features up to window size w2
    X_tr = X_train_lags[:, :w2]
    X_va = X_val_lags[:, :w2]
    for K2 in K2_range:
        knn = KNeighborsRegressor(n_neighbors=K2, n_jobs=-1)
        knn.fit(X_tr, y_train)
        pred_long[(w2, K2)] = knn.predict(X_va)

# ---------------------------------------------------------
# Step 5: Evaluate Ensembles to Find the Best Hybrid Pairs
# ---------------------------------------------------------
print(f"\nEvaluating all {len(pred_short) * len(pred_long)} combinations...")
best_mae = float('inf')
best_params = None

# Grid search by combining the precomputed predictions
for (w1, K1), p_short in pred_short.items():
    for (w2, K2), p_long in pred_long.items():

        # Calculate the ensemble prediction (average of both KNNs)
        ensemble_pred = (p_short + p_long) / 2.0

        # Calculate validation Mean Absolute Error using NumPy for raw speed
        mae = np.mean(np.abs(ensemble_pred - y_val))

        if mae < best_mae:
            best_mae = mae
            best_params = (w1, K1, w2, K2)

# ---------------------------------------------------------
# Step 6: Display the Optimization Results
# ---------------------------------------------------------
opt_w1, opt_K1, opt_w2, opt_K2 = best_params

print("\n" + "="*50)
print("GRID SEARCH COMPLETE")
print("="*50)
print(f"Best Validation MAE: {best_mae:.2f}")
print(f"Optimal Short Window (w1): {opt_w1}")
print(f"Optimal Short K-Neighbors (K1): {opt_K1}")
print(f"Optimal Long Window (w2): {opt_w2}")
print(f"Optimal Long K-Neighbors (K2): {opt_K2}")
print("="*50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Generating lag features...
Dataframe successfully prepared. Total rows available: 25897


/tmp/ipykernel_699/4133411863.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  eia_data[col_name] = eia_data['Demand'].shift(i)
/tmp/ipykernel_699/4133411863.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  eia_data[col_name] = eia_data['Demand'].shift(i)
/tmp/ipykernel_699/4133411863.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-frag

Data successfully split:
 - Train rows: 16833
 - Validation rows: 4402
 - Test rows: 4662

Precomputing short-window model predictions...
Precomputing long-window model predictions...

Evaluating all 28561 combinations...

GRID SEARCH COMPLETE
Best Validation MAE: 966.48
Optimal Short Window (w1): 3
Optimal Short K-Neighbors (K1): 3
Optimal Long Window (w2): 4
Optimal Long K-Neighbors (K2): 6
